# Parsimony: LoRA fine-tune of a small model
Run on Colab: **Runtime > Change runtime type > T4 GPU**. Run cells top to bottom (~25 min).
First upload `ml.zip` (the repo's `ml/` folder zipped) using the Files panel on the left.

It trains Qwen2.5-1.5B with LoRA on the synthetic expense data, then scores the tuned model on the
100 hand-labeled test cases, which are never used for training. The untuned base model scored 0/100,
so that step is left out. See `ml/RESULTS.md` for all numbers.

In [ ]:
!pip -q install -U transformers peft trl datasets accelerate
!pip -q uninstall -y torchao   # Colab ships an old torchao that newer peft rejects; we do not use it
!unzip -oq ml.zip
import sys, importlib; sys.path.insert(0, 'ml'); sys.path.insert(0, 'ml/eval'); importlib.invalidate_caches()
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
import json, time, datetime as dt, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from prompt import make_prompt, finalize
import score

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'   # small, Apache-2.0, knows some Korean; change here to try another
tok = AutoTokenizer.from_pretrained(BASE)
cases = score.load_cases()

def run_model(model, name):
    model.eval(); preds = []; secs = []
    for c in cases:
        p = make_prompt(c['text'], dt.date.fromisoformat(c['reference_date']), c['default_currency'])
        ids = tok(p, return_tensors='pt').to(model.device)
        t = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=220, do_sample=False, pad_token_id=tok.eos_token_id)
        secs.append(time.perf_counter() - t)
        text = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        try: obj = json.loads(text)
        except Exception: obj = None
        preds.append(finalize(obj, dt.date.fromisoformat(c['reference_date'])))  # date code -> date, category check
    print(f'== {name} ==  median {sorted(secs)[len(secs)//2]:.2f}s per answer (T4 GPU)')
    print(score.report(cases, preds))
    return preds

## 1. Train (LoRA)
LoRA freezes the model and trains small add-on matrices (well under 2% of the weights).

In [ ]:
import os
TRAIN = next(f for f in ('ml/data/train_v3.jsonl', 'ml/data/train_v2.jsonl', 'ml/data/train.jsonl') if os.path.exists(f))
print('training on', TRAIN)
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ds = load_dataset('json', data_files=TRAIN)['train'].shuffle(seed=1)
ds = ds.map(lambda r: {'prompt': r['prompt'], 'completion': ' ' + r['completion'] + tok.eos_token},
            remove_columns=['lang'])
split = ds.train_test_split(test_size=0.05, seed=1)   # 5% only to watch the loss; NOT the test set

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
cfg = SFTConfig(output_dir='out', num_train_epochs=2, per_device_train_batch_size=8,
                gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type='cosine',
                warmup_steps=20, logging_steps=20, eval_strategy='epoch', save_strategy='no',
                fp16=True, max_length=512, report_to='none')
# Free the GPU first in case this cell is re-run in the same session.
import gc
for _n in ('model', 'trainer'):
    globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32, device_map={'': 0})  # all on the GPU
trainer = SFTTrainer(model=model, args=cfg, train_dataset=split['train'], eval_dataset=split['test'],
                     processing_class=tok, peft_config=lora)
trainer.train()
trainer.model.save_pretrained('parsimony-lora')   # tiny adapter folder; zip and download it before the session ends

## 2. Evaluate on the 100 held-out cases

In [ ]:
trainer.model.half()
tuned_preds = run_model(trainer.model, 'fine-tuned (LoRA)')
rows = [{'text': c['text'], 'output': o} for c, o in zip(cases, tuned_preds)]
open('preds_tuned.jsonl','w').write('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows))

## 3. Read the result honestly
Whatever the numbers are, those are the numbers that go in the README. Then save the adapter:
`!zip -qr lora.zip parsimony-lora` and `from google.colab import files; files.download('lora.zip')`.